In [ ]:
#!/usr/bin/env python3
# Filename: manage_pipeline.py

import sys
import os
import argparse
import subprocess
from pathlib import Path
from typing import List, Dict, Any, Set

# --- パイプライン定義 ---
# 'execution_per_target': True に設定されたステップは、各サブディレクトリに対して個別に実行される
PIPELINE_STEPS: List[Dict[str, Any]] = [
    {
        'name': 'run_cdhit',
        'script': 'scripts/microbiome_cd-hit.py',
        'execution_per_target': True, # サブディレクトリ毎に実行
        'depends_on': [],
    },
    {
        'name': 'blast_screening',
        'script': 'scripts/microbiome_cdhit_blastscreening.py',
        'execution_per_target': True, # サブディレクトリ毎に実行
        'depends_on': ['run_cdhit'],
    },
    {
        'name': 'external_blast',
        'script': 'scripts/microbiome_externalblast.py',
        'execution_per_target': True, # サブディレクトリ毎に実行
        'depends_on': ['blast_screening'],
    }
]
# -----------------------------------------

def select_target_directory() -> str:
    """Interactively select a target directory."""
    print("Searching for target directories containing 'commands.txt'...")
    potential_dirs = [
        d.name for d in Path.cwd().iterdir() if d.is_dir() and (d / 'commands.txt').is_file()
    ]
    if not potential_dirs:
        print("Error: No target directories containing 'commands.txt' were found.", file=sys.stderr)
        sys.exit(1)
    print("\nPlease choose the directory to process:")
    for i, directory in enumerate(potential_dirs):
        print(f" {i + 1:2d}: {directory}")
    try:
        choice_str = input(f"\nEnter number (1-{len(potential_dirs)}): ")
        choice = int(choice_str.strip())
        if 1 <= choice <= len(potential_dirs):
            chosen_dir = potential_dirs[choice - 1]
            print(f"-> Target directory set to '{chosen_dir}'.")
            return chosen_dir
        else:
            raise ValueError("Choice out of range.")
    except (ValueError, EOFError, KeyboardInterrupt):
        print("\nError: Invalid selection or operation cancelled.", file=sys.stderr)
        sys.exit(1)

def select_target_subdirectories(sample_dir: Path) -> List[str]:
    """Interactively select subdirectories (Genus_*, Species_*) to process."""
    print(f"\nSearching for Genus_* or Species_* subdirectories in '{sample_dir.name}'...")
    
    potential_subdirs = [
        d.name for d in sample_dir.iterdir() 
        if d.is_dir() and (d.name.startswith('Genus_') or d.name.startswith('Species_'))
    ]
    
    if not potential_subdirs:
        print(f"Error: No 'Genus_*' or 'Species_*' subdirectories found in '{sample_dir.name}'.", file=sys.stderr)
        sys.exit(1)
        
    potential_subdirs.sort()

    print("Please choose the subdirectories to process:")
    for i, subdir in enumerate(potential_subdirs):
        print(f" {i + 1:2d}: {subdir}")
    
    try:
        choice_str = input(f"\nEnter number(s) separated by commas (e.g., 1,3,4), or 'all': ").strip().lower()
        
        if choice_str == 'all':
            chosen_subdirs = potential_subdirs
        else:
            choices = {int(c.strip()) for c in choice_str.split(',')}
            chosen_subdirs = [
                potential_subdirs[i - 1] for i in choices if 1 <= i <= len(potential_subdirs)
            ]
        
        if not chosen_subdirs:
             raise ValueError("No valid selection was made.")

        print("-> The following subdirectories will be processed:")
        for subdir in chosen_subdirs:
            print(f"   - {subdir}")
        return chosen_subdirs

    except (ValueError, EOFError, KeyboardInterrupt):
        print("\nError: Invalid selection or operation cancelled.", file=sys.stderr)
        sys.exit(1)

def main():
    """Main function to run the pipeline steps."""
    parser = argparse.ArgumentParser(
        description="Run a series of processing steps in a controlled pipeline.",
        formatter_class=argparse.RawTextHelpFormatter
    )
    parser.add_argument('--base-dir', type=str,
                        help="The base directory for a specific sample.")
    parser.add_argument('--cpu', type=int, default=8,
                        help="Number of CPU cores to use for applicable steps.")
    parser.add_argument('extra_args', nargs=argparse.REMAINDER,
                        help="Target subdirectories (e.g., Genus_A Genus_B). If omitted, you will be prompted.")
    
    args, unknown = parser.parse_known_args()
    
    if args.extra_args:
        args.extra_args = [arg for arg in args.extra_args if not arg.endswith('.json')]

    if not args.base_dir:
        args.base_dir = select_target_directory()
        
    project_root_dir = Path.cwd()
    sample_base_dir = Path(args.base_dir).resolve()
    
    if not sample_base_dir.is_dir():
        print(f"Error: Base directory '{sample_base_dir}' does not exist.", file=sys.stderr)
        sys.exit(1)

    if not args.extra_args:
        args.extra_args = select_target_subdirectories(sample_base_dir)
        
    done_dir = sample_base_dir / 'done'
    done_dir.mkdir(exist_ok=True)
    
    # --- ▼▼▼ 実行ロジックを全面的に更新 ▼▼▼ ---
    # 各サブディレクトリに対して、パイプラインのステップを順番に実行する
    for target in args.extra_args:
        print(f"\n\nProcessing Target: ================== {target} ==================")
        for step in PIPELINE_STEPS:
            step_name = step['name']
            
            # 1. サブディレクトリごとの.doneファイル名を定義
            flag_file = done_dir / f"{step_name}_{target}.done"

            # 2. 完了済みかチェック
            if flag_file.is_file():
                print(f"✅ Skip {step_name} for '{target}' (already done)")
                continue

            # 3. 依存関係をチェック (サブディレクトリごと)
            dependencies_met = True
            for dep in step['depends_on']:
                dep_flag_file = done_dir / f"{dep}_{target}.done"
                if not dep_flag_file.is_file():
                    print(f"⚠️ Cannot run {step_name} for '{target}': dependency '{dep}' is not completed.", file=sys.stderr)
                    dependencies_met = False
                    break
            
            if not dependencies_met:
                print(f"❌ Halting processing for target '{target}' due to missing dependencies.", file=sys.stderr)
                # このターゲットの処理を中止し、次のターゲットへ
                break 

            print(f"🚀 === STEP: {step_name} for {target} ===")
            
            script_path = project_root_dir / step['script']
            if not script_path.is_file():
                print(f"Error: Script file not found: {script_path}", file=sys.stderr)
                sys.exit(1)
            
            # 4. コマンドを組み立てる
            interpreter = 'python3' if script_path.suffix == '.py' else 'perl' if script_path.suffix == '.pl' else ''
            cmd = [interpreter, str(script_path)] if interpreter else [str(script_path)]

            # スクリプトごとに必要な引数を渡す
            if step_name == 'run_cdhit':
                cmd.extend(['--base-dir', str(sample_base_dir), target])
            elif step_name == 'blast_screening':
                cmd.extend(['--base-dir', str(sample_base_dir), '--cpu', str(args.cpu), target])
            elif step_name == 'external_blast':
                cmd.extend(['--base-dir', str(project_root_dir), '--target-dir', sample_base_dir.name, target])

            # 5. コマンドを実行
            print(f"   Running: {' '.join(cmd)}")
            try:
                subprocess.run(cmd, check=True, text=True, capture_output=False)
            except subprocess.CalledProcessError as e:
                print(f"❌ Step {step_name} for '{target}' failed (exit code: {e.returncode})", file=sys.stderr)
                # このターゲットの処理を中止し、次のターゲットへ
                break 
            except Exception as e:
                print(f"❌ An unexpected error occurred during step {step_name} for '{target}': {e}", file=sys.stderr)
                break

            # 6. .doneファイルに "done" と書き込む
            try:
                flag_file.write_text("done\n")
            except IOError as e:
                print(f"❌ Cannot write flag file {flag_file}: {e}", file=sys.stderr)
                sys.exit(1)
            
            print(f"🎉 Completed {step_name} for {target}")
    # --- ▲▲▲ ここまで更新 ▲▲▲ ---

    print("\n\n✨ All selected targets have been processed.")

if __name__ == "__main__":
    main()

Searching for target directories containing 'commands.txt'...

Please choose the directory to process:
  1: microbiome
  2: pain_sensitivity
  3: silva_pain_sensitivity
  4: depression
  5: level-7

Enter number (1-5): 4
-> Target directory set to 'depression'.

Searching for Genus_* or Species_* subdirectories in 'depression'...
Please choose the subdirectories to process:
  1: Genus_Adlercreutzia
  2: Genus_Aerococcus
  3: Genus_Allobaculum
  4: Genus_Anaerofustis
  5: Genus_Anaerostipes
  6: Genus_Anaerotruncus
  7: Genus_Arthrobacter
  8: Genus_Bacteroides
  9: Genus_Blautia
 10: Genus_Brachybacterium
 11: Genus_Brevibacterium
 12: Genus_Candidatus_Neoarthromitus
 13: Genus_Clostridium
 14: Genus_Coprobacillus
 15: Genus_Coprococcus
 16: Genus_Corynebacterium
 17: Genus_Dehalobacterium
 18: Genus_Desulfovibrio
 19: Genus_Dietzia
 20: Genus_Dorea
 21: Genus_Enterococcus
 22: Genus_Facklamia
 23: Genus_Holdemania
 24: Genus_Lactobacillus
 25: Genus_Odoribacter
 26: Genus_Oscillospira